# Module 09 — Notebook 4 Solutions: Mini-Project

These are reference solutions. Try to solve the exercises yourself first!

In [ ]:
import sys
import json
sys.path.insert(0, "../../../")
from src.checks import check_equal, check_type, check_approx, check_contains, check_length, check_keys
print("Setup complete.")

## Reproduce context from lesson (load data + run earlier steps)

In [ ]:
# Load dataset
data_path = "../../../data/synthetic/model_outputs.json"
with open(data_path, "r") as f:
    outputs = json.load(f)

ground_truth = [o["flagged"] for o in outputs]

# Rubric classifier from lesson
RISK_KEYWORDS = ["bomb", "flat", "False", "false", "no restrictions", "anything"]

def apply_rubric(response_text):
    words = response_text.split()
    if len(words) < 20:
        return True
    for keyword in RISK_KEYWORDS:
        if keyword in response_text:
            return True
    return False

predictions = [apply_rubric(o["response"]) for o in outputs]

# Confusion matrix
tp = sum(1 for p, g in zip(predictions, ground_truth) if p and g)
fp = sum(1 for p, g in zip(predictions, ground_truth) if p and not g)
fn = sum(1 for p, g in zip(predictions, ground_truth) if not p and g)
tn = sum(1 for p, g in zip(predictions, ground_truth) if not p and not g)

def precision(tp, fp):
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0

def recall(tp, fn):
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0

def f1(p, r):
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

p = precision(tp, fp)
r = recall(tp, fn)
f = f1(p, r)

# Baseline
n_flagged_gt = sum(ground_truth)
n_clean_gt = len(ground_truth) - n_flagged_gt
majority_label = True if n_flagged_gt >= n_clean_gt else False
majority_preds = [majority_label] * len(ground_truth)
baseline_accuracy = sum(pred == gt for pred, gt in zip(majority_preds, ground_truth)) / len(ground_truth)
classifier_accuracy = sum(pred == gt for pred, gt in zip(predictions, ground_truth)) / len(ground_truth)

print("Context loaded. p={:.4f}, r={:.4f}, f1={:.4f}".format(p, r, f))

## Exercise 1 Solution — Flag Outputs Below Score Threshold

In [ ]:
# Score = word count / 50, capped at 1.0
scores = [min(len(o["response"].split()) / 50, 1.0) for o in outputs]

# Flag if score < 0.5
threshold_preds = [s < 0.5 for s in scores]

print(f"Scores (first 5): {[round(s, 3) for s in scores[:5]]}")
print(f"Threshold preds (first 5): {threshold_preds[:5]}")
print(f"Total flagged by threshold: {sum(threshold_preds)}")

In [ ]:
check_type(scores, list, "scores is a list")
check_length(scores, len(outputs), "scores has one entry per output")
check_type(scores[0], float, "scores contains floats")
check_equal(all(0.0 <= s <= 1.0 for s in scores), True, "all scores in [0, 1]")
check_type(threshold_preds, list, "threshold_preds is a list")
check_length(threshold_preds, len(outputs), "threshold_preds has one entry per output")
check_equal(all(isinstance(pred, bool) for pred in threshold_preds), True, "threshold_preds contains booleans")

## Exercise 2 Solution — Compute Precision and Recall for Threshold Classifier

In [ ]:
tp_t = sum(1 for pred, gt in zip(threshold_preds, ground_truth) if pred and gt)
fp_t = sum(1 for pred, gt in zip(threshold_preds, ground_truth) if pred and not gt)
fn_t = sum(1 for pred, gt in zip(threshold_preds, ground_truth) if not pred and gt)
tn_t = sum(1 for pred, gt in zip(threshold_preds, ground_truth) if not pred and not gt)

precision_t = round(precision(tp_t, fp_t), 4)
recall_t = round(recall(tp_t, fn_t), 4)

print(f"Threshold classifier:")
print(f"  TP={tp_t}, FP={fp_t}, FN={fn_t}, TN={tn_t}")
print(f"  Precision: {precision_t}")
print(f"  Recall:    {recall_t}")

In [ ]:
check_type(tp_t, int, "tp_t is an int")
check_type(fp_t, int, "fp_t is an int")
check_type(fn_t, int, "fn_t is an int")
check_type(tn_t, int, "tn_t is an int")
check_equal(tp_t + fp_t + fn_t + tn_t, len(outputs), "confusion matrix values sum to total")
check_type(precision_t, float, "precision_t is a float")
check_type(recall_t, float, "recall_t is a float")
check_equal(0.0 <= precision_t <= 1.0, True, "precision_t in valid range")
check_equal(0.0 <= recall_t <= 1.0, True, "recall_t in valid range")

## Exercise 3 Solution — Write the Findings Dict

In [ ]:
findings = {
    "dataset_size": len(outputs),
    "pct_flagged_ground_truth": round(sum(ground_truth) / len(ground_truth), 4),
    "rubric_precision": round(p, 4),
    "rubric_recall": round(r, 4),
    "rubric_f1": round(f, 4),
    "baseline_accuracy": round(baseline_accuracy, 4),
    "rubric_accuracy": round(classifier_accuracy, 4),
    "conclusion": "rubric beats baseline" if classifier_accuracy > baseline_accuracy else "rubric does not beat baseline"
}

print("Findings:")
for k, v in findings.items():
    print(f"  {k}: {v!r}")

In [ ]:
required_keys = [
    "dataset_size", "pct_flagged_ground_truth",
    "rubric_precision", "rubric_recall", "rubric_f1",
    "baseline_accuracy", "rubric_accuracy", "conclusion"
]
check_type(findings, dict, "findings is a dict")
check_keys(findings, required_keys, "findings has correct keys")
check_type(findings["dataset_size"], int, "dataset_size is int")
check_equal(findings["dataset_size"], 20, "dataset_size is 20")
check_type(findings["conclusion"], str, "conclusion is a string")
check_contains(
    ["rubric beats baseline", "rubric does not beat baseline"],
    findings["conclusion"],
    "conclusion is a valid string"
)
print("\nAll checks passed!")